# 07 — Transfer learning: a frozen ResNet, then careful fine-tuning

**Prerequisites:** notebooks 05–06; understand a classifier head, validation, and learning rate.
**Goals:** adapt genuine ImageNet-pretrained ResNet-18 features to CIFAR-10, freeze and unfreeze
layers correctly, handle BatchNorm state, and choose checkpoints using validation only.
**Time:** 30–60 minutes including downloads and experiments. Normal mode downloads CIFAR-10
(about 170 MB archive) and pretrained weights (about 45 MB). Allow roughly 1 GB free disk space.

The quick default trains on **2,000** class-balanced images and uses **500** each for validation
and test. Batch 32 and two short stages fit a typical 8 GB GPU. This is a learning experiment,
not a benchmark result. Set `COURSE_SMOKE_TEST=1` before launching the kernel for a tiny offline
**FakeData + randomly initialized ResNet** wiring check; that mode does **not** demonstrate learned
transfer or meaningful accuracy. Normal download failures raise errors; no synthetic fallback occurs.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms as T
from course_utils import DATA_DIR, evaluate

BATCH_SIZE = 16 if SMOKE_TEST else 32
weights = models.ResNet18_Weights.DEFAULT
evaluation_transform = weights.transforms()
training_transform = T.Compose([
    T.Resize(256), T.RandomCrop(224), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(mean=evaluation_transform.mean, std=evaluation_transform.std),
])
print("Evaluation preprocessing:", evaluation_transform)
print("SMOKE: random weights and synthetic data" if SMOKE_TEST else "REAL: ImageNet weights + CIFAR-10")


## 1. Match preprocessing and keep partitions disjoint

The pretrained weights expect three-channel images, a 224×224 crop, and their ImageNet
normalization. We obtain validation preprocessing directly from the weights object and use the
same normalization during training. Resizing CIFAR-10's 32×32 images does not add new detail.

Validation comes from the official training partition; final test images come from the official
test partition. We explicitly select equal counts per class for this quick experiment.


In [ ]:
def balanced_indices(targets, per_class, seed=42, exclude=()):
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    allowed = np.ones(len(targets), dtype=bool)
    allowed[np.asarray(list(exclude), dtype=int)] = False
    result = []
    for label in np.unique(targets):
        pool = np.flatnonzero((targets == label) & allowed)
        if len(pool) < per_class:
            raise ValueError(f"Insufficient examples of class {label}")
        result.extend(rng.permutation(pool)[:per_class].tolist())
    rng.shuffle(result)
    return result

if SMOKE_TEST:
    # Explicit offline verification only: no learned pretrained features are loaded.
    train_data = datasets.FakeData(32, (3, 32, 32), 10, transform=training_transform, random_offset=0)
    val_data = datasets.FakeData(16, (3, 32, 32), 10, transform=evaluation_transform, random_offset=1000)
    test_data = datasets.FakeData(16, (3, 32, 32), 10, transform=evaluation_transform, random_offset=2000)
    class_names = [f"synthetic_{i}" for i in range(10)]
    split_indices = {"mode": "synthetic wiring verification"}
else:
    base_train = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=training_transform)
    base_val = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=evaluation_transform)
    base_test = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=evaluation_transform)
    train_ids = balanced_indices(base_train.targets, 200, seed=42)
    val_ids = balanced_indices(base_train.targets, 50, seed=43, exclude=train_ids)
    test_ids = balanced_indices(base_test.targets, 50, seed=44)
    assert not set(train_ids) & set(val_ids)
    train_data, val_data, test_data = (Subset(base_train, train_ids),
                                      Subset(base_val, val_ids), Subset(base_test, test_ids))
    class_names = base_train.classes
    split_indices = {"train": train_ids, "val": val_ids, "test": test_ids}

generator = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=generator)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Train/val/test:", len(train_data), len(val_data), len(test_data))


In [ ]:
xb, yb = next(iter(train_loader))
mean = torch.tensor(evaluation_transform.mean).view(3, 1, 1)
std = torch.tensor(evaluation_transform.std).view(3, 1, 1)
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, image, label in zip(axes, xb[:6], yb[:6]):
    ax.imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(class_names[label.item()], fontsize=9); ax.axis("off")
plt.show()


## 2. Replace the classifier and freeze the backbone

The original ImageNet head predicts 1,000 categories. Our new randomly initialized head predicts
ten categories. First train only this head so its early gradients cannot disturb useful features.

**Subtle point:** `requires_grad=False` freezes parameters, but does not stop BatchNorm running
averages from changing in `train()` mode. During head training, keep the entire model in `eval()`
and put just the head in `train()`. During fine-tuning, freeze BatchNorm statistics explicitly.
This is a useful small-dataset strategy; with abundant data, updating statistics can be reasonable.


In [ ]:
seed_everything(42)
model = models.resnet18(weights=None if SMOKE_TEST else weights)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

def show_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable:,} / {total:,}")

show_trainable(model)


In [ ]:
def set_stage_mode(model, stage):
    if stage == "head":
        model.eval()
        model.fc.train()
    else:
        model.train()
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()  # Keep the pretrained running means/variances fixed.

def train_stage(model, optimizer, epochs, stage):
    history, best_loss, best_state = [], float("inf"), None
    for epoch in range(1, epochs + 1):
        set_stage_mode(model, stage)
        total_loss, correct, count = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
        validation = evaluate(model, val_loader, device=device)
        row = dict(stage=stage, epoch=epoch, train_loss=total_loss/count,
                   train_acc=correct/count, val_loss=validation["loss"], val_acc=validation["accuracy"])
        history.append(row)
        if row["val_loss"] < best_loss:
            best_loss = row["val_loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(row)
    model.load_state_dict(best_state)
    return history

head_optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
head_history = train_stage(model, head_optimizer, 1 if SMOKE_TEST else 3, "head")
head_validation = evaluate(model, val_loader, device=device)
head_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


## 3. Fine-tune the final residual block gently

Unfreeze `layer4`, then create a **new optimizer** containing those newly trainable parameters.
Its learning rate is ten times smaller than the head's: pretrained features usually need smaller
updates. Earlier blocks remain frozen. BatchNorm affine parameters remain frozen too, while
`set_stage_mode` also freezes their running statistics.

A lower training loss after unfreezing does not guarantee a better validation result. We retain
the head-only checkpoint in case fine-tuning makes validation worse.


In [ ]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True
for module in model.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        for parameter in module.parameters():
            parameter.requires_grad = False
fine_optimizer = torch.optim.AdamW([
    {"params": [p for p in model.layer4.parameters() if p.requires_grad], "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
show_trainable(model)
bn_before = model.bn1.running_mean.detach().clone()
fine_history = train_stage(model, fine_optimizer, 1 if SMOKE_TEST else 2, "fine_tune")
assert torch.equal(bn_before, model.bn1.running_mean), "Frozen BatchNorm statistics changed"
fine_validation = evaluate(model, val_loader, device=device)


In [ ]:
history_df = pd.DataFrame(head_history + fine_history)
history_df["global_epoch"] = np.arange(1, len(history_df) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(history_df.global_epoch, history_df.train_loss, label="train")
axes[0].plot(history_df.global_epoch, history_df.val_loss, label="validation")
axes[1].plot(history_df.global_epoch, history_df.val_acc, label="validation accuracy")
for ax in axes:
    ax.axvline(len(head_history) + 0.5, color="gray", linestyle="--")
    ax.set_xlabel("Epoch (dashed line: start fine-tuning)"); ax.legend()
plt.tight_layout(); plt.show()
print("Head validation:", head_validation["accuracy"], "Fine-tuned:", fine_validation["accuracy"])


## 4. Select on validation, evaluate once on test, and save

The curve contains all epochs; the current model contains the best fine-tuning epoch. Compare
that checkpoint with the best head-only checkpoint using validation loss, then evaluate the winner
on the test set. Synthetic smoke-test scores have no learning interpretation.


In [ ]:
selected_stage = "fine_tune"
if head_validation["loss"] <= fine_validation["loss"]:
    model.load_state_dict(head_state)
    selected_stage = "head"
test_result = evaluate(model, test_loader, device=device)
print("Selected stage:", selected_stage, "| test accuracy:", f"{test_result['accuracy']:.1%}")
checkpoint = {
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "architecture": "resnet18", "classes": class_names, "selected_stage": selected_stage,
    "pretrained_source": None if SMOKE_TEST else str(weights),
    "smoke_test": SMOKE_TEST, "image_size": 224,
    "mean": evaluation_transform.mean, "std": evaluation_transform.std,
}
suffix = "smoke" if SMOKE_TEST else "cifar10"
torch.save(checkpoint, ARTIFACTS / f"07_resnet18_{suffix}.pt")
(ARTIFACTS / f"07_splits_{suffix}.json").write_text(json.dumps(split_indices), encoding="utf-8")


### Exercise: Audit frozen parameters

List trainable parameter names. Confirm earlier blocks and BatchNorm parameters are frozen.


In [ ]:
trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
print("\n".join(trainable_names))
# TODO: Assert that every name starts with layer4. or fc.


<details><summary>Sample answer — open after trying</summary>

```python
assert all(name.startswith(('layer4.', 'fc.')) for name in trainable_names)
assert all(not p.requires_grad for m in model.modules()
           if isinstance(m, nn.modules.batchnorm._BatchNorm) for p in m.parameters())
```

</details>


### Exercise: Design a fair transfer comparison

Create a randomly initialized comparison model. Write down what must stay fixed before optionally training it.


In [ ]:
seed_everything(42)
scratch_model = models.resnet18(weights=None, num_classes=10)
print("Scratch parameters:", sum(p.numel() for p in scratch_model.parameters()))
# TODO: Plan a comparison with identical data splits and compute budget.
del scratch_model


<details><summary>Sample answer — open after trying</summary>

Keep splits, preprocessing, seed policy, batch size, selection metric, and training budget fixed. A short budget favors the pretrained model; report that limitation. A fair best-performance comparison requires separate validation tuning and often more scratch-training compute.

</details>


### Exercise: Read optimizer groups

Inspect both learning rates and the number of parameters they control. Explain why changing requires_grad alone would not update an already-created head-only optimizer.


In [ ]:
for i, group in enumerate(fine_optimizer.param_groups):
    print("Group", i, "lr", group["lr"], "parameters", sum(p.numel() for p in group["params"]))
# TODO: Predict what changes if the backbone LR is raised to 0.01.


<details><summary>Sample answer — open after trying</summary>

An optimizer holds the parameter objects supplied at creation. Newly unfrozen objects must be added or included in a new optimizer. A very large backbone LR can rapidly overwrite pretrained features; validation must test the effect.

</details>


**Checkpoint:** explain why frozen BatchNorm needs explicit mode handling, when to recreate an
optimizer, and why normalization must match the pretrained weights.

Next: [08 — Your own image dataset](08_your_own_image_dataset.ipynb).
References: [ResNet-18 weights and preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html),
[official transfer tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html),
[CIFAR-10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html).
